# RF-DETR: DINOv2-backbone detector on the diverse EV dataset (GPU)

The modern 'accurate + general from little data' route. RF-DETR (Roboflow, 2025)
is a real-time DETR with a **frozen DINOv2 self-supervised backbone** — exactly the
recipe the 2024-2026 research points to for generalization in small-data regimes
(frozen foundation backbone + light detection head).

Trains on the diverse multi-source data and evaluates on the 225-image diverse test.
Compare against: MTech specialist 0.277, and your YOLO11n generalist number.

Upload `ev_diverse_data.zip` to Drive; the notebook finds it. Needs a GPU runtime.

In [ ]:
!pip install -q "rfdetr[train,loggers]" supervision
import torch; print('CUDA:', torch.cuda.is_available())

In [ ]:
# Get data
from google.colab import drive
import glob, subprocess
drive.mount('/content/drive')
hits = glob.glob('/content/drive/MyDrive/**/ev_diverse_data.zip', recursive=True)
assert hits, 'ev_diverse_data.zip not found in Drive.'
subprocess.run(['unzip','-q','-o',hits[0],'-d','/content/'])
print('unzipped', hits[0])

In [ ]:
# Convert YOLO -> COCO (RF-DETR format): dataset/{train,valid,test}/_annotations.coco.json + images
import json, shutil
from pathlib import Path
from PIL import Image
SRC = Path('/content/ev_diverse'); DST = Path('/content/ev_coco')
split_map = {'train':'train','val':'valid','test':'test'}
cats = [{'id':0,'name':'module','supercategory':'none'},{'id':1,'name':'busbar','supercategory':'none'}]
for ys, cs in split_map.items():
    outd = DST/cs; outd.mkdir(parents=True, exist_ok=True)
    images, anns, aid = [], [], 1
    imgs = sorted((SRC/'images'/ys).glob('*.jpg')) + sorted((SRC/'images'/ys).glob('*.png')) + sorted((SRC/'images'/ys).glob('*.jpeg'))
    for iid, ip in enumerate(imgs, 1):
        w,h = Image.open(ip).size
        shutil.copy(ip, outd/ip.name)
        images.append({'id':iid,'file_name':ip.name,'width':w,'height':h})
        lp = SRC/'labels'/ys/(ip.stem+'.txt')
        if lp.exists():
            for line in lp.read_text().splitlines():
                p=line.split()
                if len(p)!=5: continue
                c,cx,cy,bw,bh = int(float(p[0])),*map(float,p[1:])
                x,y,bw,bh = (cx-bw/2)*w,(cy-bh/2)*h,bw*w,bh*h
                anns.append({'id':aid,'image_id':iid,'category_id':c,'bbox':[x,y,bw,bh],'area':bw*bh,'iscrowd':0})
                aid+=1
    json.dump({'images':images,'annotations':anns,'categories':cats}, open(outd/'_annotations.coco.json','w'))
    print(cs, len(images), 'images', len(anns), 'annotations')

In [ ]:
# Train RF-DETR (nano). 35 epochs = fast first pass from the COCO checkpoint.
from rfdetr import RFDETRNano
model = RFDETRNano()
model.train(dataset_dir='/content/ev_coco', epochs=35, batch_size=8, grad_accum_steps=2,
            lr=1e-4, output_dir='/content/rfdetr_out')

In [ ]:
# Evaluate on the diverse test set (mAP) using supervision
import supervision as sv
from rfdetr import RFDETRNano
ds = sv.DetectionDataset.from_coco('/content/ev_coco/test',
        '/content/ev_coco/test/_annotations.coco.json')
m = RFDETRNano(pretrain_weights='/content/rfdetr_out/checkpoint_best_ema.pth')
targets, preds = [], []
for path, image, ann in ds:
    import numpy as np
    from PIL import Image as PImage
    p = m.predict(PImage.open(path).convert('RGB'), threshold=0.3)
    targets.append(ann); preds.append(p)
mrec = sv.MeanAveragePrecision.from_detections(preds, targets)
print(f'RF-DETR diverse-test mAP50={mrec.map50:.3f}  mAP50-95={mrec.map50_95:.3f}')
print('compare: MTech specialist 0.277, YOLO11n generalist (your number)')

In [ ]:
from google.colab import files
import shutil
shutil.copy('/content/rfdetr_out/checkpoint_best_ema.pth', '/content/drive/MyDrive/ev_rfdetr_best.pth')
print('saved to Drive: MyDrive/ev_rfdetr_best.pth')
files.download('/content/rfdetr_out/checkpoint_best_ema.pth')

## Read the result
RF-DETR's DINOv2 backbone should generalize better across pack types than a
from-scratch YOLO, especially on the diverse test. Send me the mAP and we compare
the three detectors (specialist / YOLO11n generalist / RF-DETR) head to head.